In [3]:
 import pandas as pd


In [4]:
df=pd.read_csv("IMDB Dataset.csv")


In [5]:
df.shape

(50000, 2)

In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.shape
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Pre Processing

## 1 Converting to lowercase


In [9]:
df["review"]=df["review"].str.lower()

## 2 remove the urls

In [10]:
import re

def remove_urls(text):
    text=re.sub(r"https\S+" , "", text)# (pattern,repl,string)
    return text
df["review"]=df["review"].apply(remove_urls)


## 3 Remove punctuations


In [11]:
def remove_punc(text):
    text=re.sub(r"[^A-Za-z0-9\s]" , "", text)#A-Z,a-z,0-9 
    return text
df["review"]=df["review"].apply(remove_punc)    

## 4 removing html


In [12]:
def remove_html(text):
    text=re.sub(r"<.*?>" , "" , text)
    return text

df["review"]=df["review"].apply(remove_html)    

## 5 Removing Stopwords

In [13]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Kavya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Kavya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Kavya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
sample_text="I like coding in python!!"
tokens=word_tokenize(sample_text)
tokens


['I', 'like', 'coding', 'in', 'python', '!', '!']

In [16]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")# english language ke saare stopwords aajanenge
# now apan ko check krna hai ki kya vo sdtopword apne token me hai kya agar hai to apanko usko remove krna pdega
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text
    
df["review"] = df["review"].apply(remove_stopwords)     

In [17]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


## 6 Stemming

In [18]:
from nltk.stem import PorterStemmer

In [19]:
def stemming(text):#root word pe leke jaao
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_tokens=ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)    
    

In [20]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


## 7 Encoding

In [21]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])


In [22]:
y=df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

## 8 Vectorization

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [24]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057154 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3537)	0.05515484681268887
  (0, 2867)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3001)	0.47200539890110405
  (0, 1274)	0.1357698919196183
  (0, 2287)	0.049500237517476293
  (0, 1931)	0.0791260308382
  (0, 3549)	0.0963974330192639
  (0, 1360)	0.06162489377343992
  (0, 1961)	0.061560444992697486
  (0, 218)	0.08588920995304898
  (0, 1618)	0.0738170550485134
  (0, 4368)	0.041994187696759305
  (0, 4170)	0.17799685402440263
  (0, 3692)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3804)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1737)	0.037520883911174724
  (0, 4496)	0.07614066339174266
  (0, 3856)	0.17537900435282314
  (0, 1628)	0.06142445471882175
  (0, 1860)	0.07433134577032253
  (0, 3328)	0.06406818508428483
  (0, 3331)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1540)	0.17584072573791829

# DataSets and Dataloaders


In [25]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42)

In [26]:
import torch
from torch.utils.data import TensorDataset,DataLoader


In [27]:
X_train

#sparse matrix hai ye apanko isko numpy me convert krna pdega

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248306 stored elements and shape (39665, 5000)>

In [28]:
X_train=X_train.toarray()
X_test=X_test.toarray()



In [29]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
                     
)
# Now apne datasets are created and apan ab unko load krenge


In [30]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

# Build our RNN

In [31]:
import torch.nn as nn
import torch.optim as optim

In [32]:
class RNN(nn.Module):
    def  __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers


        # RNN layer
        self.rnn=nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        #fully connected layer
        #this is many to one architecture so input many=hidden size but output=1
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        #shape=(num of layers,batch_size,hidden size)
        #optional = hidden state ko apan 0 se initialise krte hai
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out, _=self.rnn(x,h0)
    #1st value= hidden state of all timesteps=> (batch,seq_length,hidden_size)
    #2nd value= final hidden state of last timestep
        out= self.fc(out[:,-1,:])
        return out
    

In [33]:
input_size=X_train.shape[1]

model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

# Training the model

In [34]:
epochs=10

for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb=Xb.unsqueeze(1)# add singleton direction

        outputs=model(Xb)#(batch_size,1)

        outputs=torch.sigmoid(outputs.squeeze()) #(batch_size,)=>prob

        loss=criterion(outputs,yb)#compute loss
        loss.backward()# backprop
        optimizer.step()#weights update

    print(f"epoch={epoch+1}/{epochs} and loss={loss.item()}")  
        
        
    

epoch=1/10 and loss=0.41819801926612854
epoch=2/10 and loss=0.36966657638549805
epoch=3/10 and loss=0.22586607933044434
epoch=4/10 and loss=0.1812611073255539
epoch=5/10 and loss=0.24933363497257233
epoch=6/10 and loss=0.23626676201820374
epoch=7/10 and loss=0.38023290038108826
epoch=8/10 and loss=0.2397860884666443
epoch=9/10 and loss=0.11611978709697723
epoch=10/10 and loss=0.20697227120399475


# Evaluate

In [37]:
model.eval()

with torch.no_grad():
    correct_vals=0
    tot_vals=0

    for Xb,yb in test_loader:
        Xb=Xb.unsqueeze(1)

        outputs=model(Xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals+=yb.size(0)
        correct_vals+=(predicted==yb).sum().item()
        #.item() python into tensor
    print(f"accuracy={correct_vals/tot_vals*100}")

accuracy=85.48956337602097
